In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

In [3]:
import os

In [32]:
import re

In [4]:
working_directory="/content/drive/MyDrive/transformers/Restaurant/Generate_Names"

In [5]:
os.chdir(working_directory)

In [6]:
os.getcwd()

'/content/drive/MyDrive/transformers/Restaurant/Generate_Names'

In [7]:
!ls

final_model		generate_namesipynb.ipynb  results
fine_tuning_GPT2.ipynb	restaurant.txt


In [11]:
print(os.listdir("./final_model"))

['config.json', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin', 'generation_config.json', 'model.safetensors']


In [ ]:
model= GPT2LMHeadModel.from_pretrained("./final_model")
tokenizer= GPT2Tokenizer.from_pretrained("./final_model")

In [13]:
input_text="The name of a new creative and funny meal is: "

In [14]:
input_ids = tokenizer.encode(input_text, return_tensors="pt")

In [15]:
input_ids

tensor([[ 464, 1438,  286,  257,  649, 7325,  290, 8258, 9799,  318,   25,  220]])

In [20]:
input_ids[0][0].item()

464

In [63]:
output = model.generate(
    input_ids,                      # starting text (your prompt in token form)

    max_length=40,                  # maximum total length (prompt + generated text)

    num_return_sequences=10,         # how many different outputs you want ( 5 dish names)

    temperature=0.8,                # controls creativity:
                                   # lower = safer / higher = more random

    top_k=50,                       # limits choices to top 50 most likely next words

    top_p=0.9,                      # nucleus sampling:
                                   # choose from smallest set of words covering 90% probability

    repetition_penalty=1.5,         # discourages repeating the same words again and again

    do_sample=True,                 # enables randomness (otherwise it's deterministic)

    pad_token_id=tokenizer.eos_token_id,  # used for padding (GPT-2 doesn't have a pad token, so we reuse EOS)
)

In [25]:
type(output)

torch.Tensor

In [31]:
tokenizer.decode(output[4], skip_special_tokens=True)

'The name of a new creative and funny meal is: iced milkshake, oreo | Cuisine American Chicken Chaos Baby Feast Roll Model Deluxe Burger Boss Sandwich Unchained Steak & Shake It'

In [52]:
def filter_non_english_words(text):
  return re.sub(r"[^A-Za-z0-9|:\s]", '', text)

In [53]:
filter_non_english_words(tokenizer.decode(output[2], skip_special_tokens=True))

'The name of a new creative and funny meal is:   | Name the Day Fries Reloaded Burger Boss Chicken Combo Meal Night Fish Fry Fever'

In [39]:
filter_non_english_words(tokenizer.decode(output[1], skip_special_tokens=True))

'The name of a new creative and funny meal is: s Delightful Morning Breakfast Dream Chicken Combo Burger Edition | Ingredients chili sauce'

In [47]:
def pick_up_the_meal_name(text):
    pattern = r"The name of a new creative and funny meal is:\s*(.+?)(?:\s*\||$)"
    match = re.search(pattern, text)
    return match.group(1) if match else None

In [48]:
text = "The name of a new creative and funny meal is: s Delightful Morning Breakfast Dream Chicken Combo Burger Edition | Ingredients chili sauce"

In [49]:
pick_up_the_meal_name(text)

's Delightful Morning Breakfast Dream Chicken Combo Burger Edition'

In [54]:
text = "The name of a new creative and funny meal is: Cuisine American Hard Day Night Fish Combo Burger Edition | Ingredients Chicken, Parmesan Cheese & Furious"

In [55]:
pick_up_the_meal_name(text)

'Cuisine American Hard Day Night Fish Combo Burger Edition'

In [69]:
def show_generated_name(output):
  for meal in output:
    decoded_output = tokenizer.decode(meal, skip_special_tokens=True)
    filtered_decoded_ouput= filter_non_english_words(decoded_output)
    meal_name= pick_up_the_meal_name(filtered_decoded_ouput)
    if meal_name:
       print(meal_name.strip("|").strip())


In [70]:
show_generated_name(output)


iced milkshake oreo
iced chicken honey
iced milkshake
Name the Day F
iced milkshake oreo
iced milkshake oreo
Cuisine American
iced milkshake oreo
Name Daybreak Deluxe Steak  Shake
